<a href="https://colab.research.google.com/github/AchiengMary/nav2/blob/master/neural_manifold_robot_control.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Neural Manifold Robot Control
## Comparing Full-Dimensional vs Manifold-Constrained Control

This notebook implements a minimal computational testbed for translating neural manifold theories into robotic control using a 6-DOF manipulator.

## 1. Setup and Dependencies

In [ ]:
# Install required packages
!pip install pybullet numpy matplotlib scipy scikit-learn pandas seaborn -q

import pybullet as p
import pybullet_data
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.spatial.transform import Rotation
from sklearn.decomposition import PCA
import pandas as pd
from typing import List, Tuple, Dict
import time
from dataclasses import dataclass
import json

# Set random seed for reproducibility
np.random.seed(42)

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 80.5/80.5 MB 15.0 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done


## 2. Robot Setup - 6-DOF Manipulator

In [ ]:
class Robot6DOF:
    """6-DOF Robot Manipulator using PyBullet"""

    def __init__(self, gui=True):
        """Initialize the robot in PyBullet simulation"""
        # Connect to PyBullet
        if gui:
            self.client = p.connect(p.GUI)
        else:
            self.client = p.connect(p.DIRECT)

        p.setAdditionalSearchPath(pybullet_data.getDataPath())
        p.setGravity(0, 0, -9.81)

        # Load plane and robot
        self.plane_id = p.loadURDF("plane.urdf")

        # Use Kuka IIWA robot (7-DOF, we'll use 6)
        self.robot_id = p.loadURDF("kuka_iiwa/model.urdf",
                                   basePosition=[0, 0, 0],
                                   useFixedBase=True)

        # Get joint information
        self.num_joints = p.getNumJoints(self.robot_id)
        self.controlled_joints = [0, 1, 2, 3, 4, 5]  # Use first 6 joints
        self.n_dof = len(self.controlled_joints)

        # Joint limits
        self.joint_limits = self._get_joint_limits()

        # End effector link
        self.ee_link = 6

        print(f"✓ Robot initialized with {self.n_dof} DOF")
        print(f"  Joint limits: {self.joint_limits}")

    def _get_joint_limits(self):
        """Get joint limits for controlled joints"""
        limits = {'lower': [], 'upper': []}
        for i in self.controlled_joints:
            info = p.getJointInfo(self.robot_id, i)
            limits['lower'].append(info[8])
            limits['upper'].append(info[9])
        return {
            'lower': np.array(limits['lower']),
            'upper': np.array(limits['upper'])
        }

    def get_joint_positions(self):
        """Get current joint positions"""
        joint_states = p.getJointStates(self.robot_id, self.controlled_joints)
        return np.array([state[0] for state in joint_states])

    def get_joint_velocities(self):
        """Get current joint velocities"""
        joint_states = p.getJointStates(self.robot_id, self.controlled_joints)
        return np.array([state[1] for state in joint_states])

    def get_ee_position(self):
        """Get end effector position"""
        state = p.getLinkState(self.robot_id, self.ee_link)
        return np.array(state[0])  # Position

    def get_ee_pose(self):
        """Get end effector position and orientation"""
        state = p.getLinkState(self.robot_id, self.ee_link)
        return np.array(state[0]), np.array(state[1])  # Position, Orientation

    def set_joint_positions(self, positions, velocity_control=False):
        """Set joint positions using position control"""
        positions = np.clip(positions,
                          self.joint_limits['lower'],
                          self.joint_limits['upper'])

        if velocity_control:
            p.setJointMotorControlArray(
                self.robot_id,
                self.controlled_joints,
                p.POSITION_CONTROL,
                targetPositions=positions,
                forces=[100] * self.n_dof
            )
        else:
            for i, joint_idx in enumerate(self.controlled_joints):
                p.resetJointState(self.robot_id, joint_idx, positions[i])

    def step(self):
        """Step the simulation"""
        p.stepSimulation()

    def reset(self):
        """Reset robot to home position"""
        home_position = np.zeros(self.n_dof)
        self.set_joint_positions(home_position)

    def disconnect(self):
        """Disconnect from PyBullet"""
        p.disconnect()

## 3. Controller Implementations

In [ ]:
@dataclass
class ControllerMetrics:
    """Store metrics for controller evaluation"""
    name: str
    dimensionality: int
    trajectory_error: List[float]
    smoothness: float
    energy: float
    completion_time: float
    success_rate: float


class BaseController:
    """Base class for all controllers"""

    def __init__(self, robot: Robot6DOF, name: str):
        self.robot = robot
        self.name = name
        self.trajectory_history = []
        self.velocity_history = []
        self.command_history = []

    def reset_history(self):
        """Reset tracking history"""
        self.trajectory_history = []
        self.velocity_history = []
        self.command_history = []

    def compute_control(self, target, current_state):
        """Compute control command - to be implemented by subclasses"""
        raise NotImplementedError

    def execute_trajectory(self, targets, dt=0.01, max_steps=1000):
        """Execute a trajectory of targets"""
        self.reset_history()

        for target in targets:
            steps = 0
            while steps < max_steps:
                current_pos = self.robot.get_joint_positions()
                current_vel = self.robot.get_joint_velocities()

                # Compute control
                control = self.compute_control(target, current_pos)

                # Apply control
                self.robot.set_joint_positions(control, velocity_control=True)
                self.robot.step()

                # Record history
                self.trajectory_history.append(current_pos.copy())
                self.velocity_history.append(current_vel.copy())
                self.command_history.append(control.copy())

                # Check convergence
                error = np.linalg.norm(current_pos - target)
                if error < 0.01:
                    break

                steps += 1
                time.sleep(dt)


class FullDimensionalController(BaseController):
    """6D Full-Dimensional Controller - Direct joint control"""

    def __init__(self, robot: Robot6DOF, gain=5.0):
        super().__init__(robot, "Full-6D")
        self.gain = gain
        self.dimensionality = 6

    def compute_control(self, target, current_state):
        """proportional control in full 6D space"""
        error = target - current_state
        control = current_state + self.gain * error * 0.01
        return control


class ManifoldController(BaseController):
    """Manifold-Constrained Controller - Operates in low-D latent space"""

    def __init__(self, robot: Robot6DOF, latent_dim=2,
                 manifold_type='linear', gain=5.0):
        super().__init__(robot, f"Manifold-{latent_dim}D-{manifold_type}")
        self.latent_dim = latent_dim
        self.manifold_type = manifold_type
        self.gain = gain
        self.dimensionality = latent_dim

        # Initialize manifold structure
        self._initialize_manifold()

    def _initialize_manifold(self):
        """Initialize manifold mapping from latent to joint space"""
        if self.manifold_type == 'linear':
            # Linear manifold using PCA-like random projection
            # Create random orthonormal basis for latent space
            self.basis = np.random.randn(6, self.latent_dim)
            self.basis, _ = np.linalg.qr(self.basis)

        elif self.manifold_type == 'synergy':
            # Synergy-based manifold (coordinated joint patterns)
            self.basis = self._create_synergy_basis()

        elif self.manifold_type == 'pca':
            # PCA-based manifold from random movements
            self.basis = self._create_pca_basis()

    def _create_synergy_basis(self):
        """Create synergy-based manifold (coordinated patterns)"""
        synergies = []

        if self.latent_dim >= 1:
            # Synergy 1: Shoulder + elbow coordination
            synergies.append([1, 0.5, 0, 0, 0, 0])

        if self.latent_dim >= 2:
            # Synergy 2: Wrist orientation
            synergies.append([0, 0, 1, 0.8, 0, 0])

        if self.latent_dim >= 3:
            # Synergy 3: Full arm extension
            synergies.append([0.3, 0.3, 0.3, 0, 0, 0])

        basis = np.array(synergies).T
        # Orthonormalize
        basis, _ = np.linalg.qr(basis)
        return basis

    def _create_pca_basis(self):
        """Create PCA-based manifold from sample movements"""
        # Generate random valid joint configurations
        n_samples = 1000
        samples = []
        for _ in range(n_samples):
            q = np.random.uniform(
                self.robot.joint_limits['lower'],
                self.robot.joint_limits['upper']
            )
            samples.append(q)

        samples = np.array(samples)

        # Apply PCA
        pca = PCA(n_components=self.latent_dim)
        pca.fit(samples)

        return pca.components_.T

    def joint_to_latent(self, joint_pos):
        """Project joint position to latent space"""
        return self.basis.T @ joint_pos

    def latent_to_joint(self, latent_pos):
        """Map latent position to joint space"""
        return self.basis @ latent_pos

    def compute_control(self, target, current_state):
        """Compute control in latent space, then map to joint space"""
        # Project current state and target to latent space
        current_latent = self.joint_to_latent(current_state)
        target_latent = self.joint_to_latent(target)

        # Compute control in latent space
        error_latent = target_latent - current_latent
        control_latent = current_latent + self.gain * error_latent * 0.01

        # Map back to joint space
        control_joints = self.latent_to_joint(control_latent)

        # Combine with current position (manifold constraint)
        # This ensures movement stays on manifold
        alpha = 0.9  # Manifold adherence strength
        control = alpha * control_joints + (1 - alpha) * current_state

        return control

## 4. Task Definitions

In [ ]:
class TaskGenerator:
    """Generate reaching and trajectory following tasks"""

    def __init__(self, robot: Robot6DOF):
        self.robot = robot

    def generate_reaching_targets(self, n_targets=10):
        """Generate random valid reaching targets"""
        targets = []
        for _ in range(n_targets):
            target = np.random.uniform(
                self.robot.joint_limits['lower'] * 0.8,
                self.robot.joint_limits['upper'] * 0.8
            )
            targets.append(target)
        return targets

    def generate_circular_trajectory(self, n_points=20, radius=0.5):
        """Generate circular trajectory in joint space"""
        center = np.zeros(6)
        trajectory = []

        for i in range(n_points):
            angle = 2 * np.pi * i / n_points
            # Create circular motion in first 2 joints
            point = center.copy()
            point[0] = radius * np.cos(angle)
            point[1] = radius * np.sin(angle)
            trajectory.append(point)

        return trajectory

    def generate_figure_eight_trajectory(self, n_points=40, scale=0.4):
        """Generate figure-8 trajectory"""
        trajectory = []

        for i in range(n_points):
            t = 2 * np.pi * i / n_points
            point = np.zeros(6)
            point[0] = scale * np.sin(t)
            point[1] = scale * np.sin(t) * np.cos(t)
            point[2] = scale * 0.5 * np.sin(2 * t)
            trajectory.append(point)

        return trajectory

    def generate_reaching_sequence(self, n_targets=5):
        """Generate sequence of reaching movements"""
        targets = self.generate_reaching_targets(n_targets)
        # Add home position at start
        return [np.zeros(6)] + targets

## 5. Evaluation Metrics

In [ ]:
class PerformanceEvaluator:
    """Evaluate controller performance across multiple metrics"""

    @staticmethod
    def compute_trajectory_error(trajectory, targets):
        """Compute tracking error over trajectory"""
        errors = []
        trajectory = np.array(trajectory)

        # For each target, find closest trajectory point
        for target in targets:
            distances = np.linalg.norm(trajectory - target, axis=1)
            min_error = np.min(distances)
            errors.append(min_error)

        return errors

    @staticmethod
    def compute_smoothness(velocities):
        """Compute smoothness using jerk (derivative of acceleration)"""
        velocities = np.array(velocities)

        # Compute acceleration
        accelerations = np.diff(velocities, axis=0)

        # Compute jerk
        jerks = np.diff(accelerations, axis=0)

        # Smoothness metric: negative log of jerk magnitude
        jerk_magnitude = np.linalg.norm(jerks, axis=1)
        smoothness = -np.log(np.mean(jerk_magnitude) + 1e-10)

        return smoothness

    @staticmethod
    def compute_energy(commands):
        """Compute energy expenditure (sum of squared commands)"""
        commands = np.array(commands)
        energy = np.sum(np.linalg.norm(commands, axis=1)**2)
        return energy

    @staticmethod
    def compute_success_rate(trajectory, targets, threshold=0.05):
        """Compute percentage of targets reached within threshold"""
        errors = PerformanceEvaluator.compute_trajectory_error(trajectory, targets)
        success_count = sum(1 for e in errors if e < threshold)
        return success_count / len(targets)

    @staticmethod
    def evaluate_controller(controller, targets):
        """Comprehensive evaluation of a controller"""
        start_time = time.time()

        # Execute trajectory
        controller.execute_trajectory(targets, dt=0.001, max_steps=500)

        completion_time = time.time() - start_time

        # Compute metrics
        trajectory_error = PerformanceEvaluator.compute_trajectory_error(
            controller.trajectory_history, targets
        )

        smoothness = PerformanceEvaluator.compute_smoothness(
            controller.velocity_history
        )

        energy = PerformanceEvaluator.compute_energy(
            controller.command_history
        )

        success_rate = PerformanceEvaluator.compute_success_rate(
            controller.trajectory_history, targets
        )

        return ControllerMetrics(
            name=controller.name,
            dimensionality=controller.dimensionality,
            trajectory_error=trajectory_error,
            smoothness=smoothness,
            energy=energy,
            completion_time=completion_time,
            success_rate=success_rate
        )

## 6. Visualization Tools

In [ ]:
class Visualizer:
    """Visualization tools for results"""

    @staticmethod
    def plot_trajectories(controllers_data, targets):
        """Plot joint space trajectories"""
        fig, axes = plt.subplots(2, 3, figsize=(15, 8))
        axes = axes.flatten()

        for joint_idx in range(6):
            ax = axes[joint_idx]

            for ctrl_name, trajectory in controllers_data.items():
                traj_array = np.array(trajectory)
                ax.plot(traj_array[:, joint_idx], label=ctrl_name, alpha=0.7)

            # Plot targets
            for i, target in enumerate(targets):
                ax.axhline(y=target[joint_idx], color='r',
                          linestyle='--', alpha=0.3, linewidth=1)

            ax.set_title(f'Joint {joint_idx + 1}')
            ax.set_xlabel('Time steps')
            ax.set_ylabel('Position (rad)')
            ax.legend()
            ax.grid(True, alpha=0.3)

        plt.tight_layout()
        return fig

    @staticmethod
    def plot_performance_comparison(metrics_list):
        """Plot performance comparison across controllers"""
        fig, axes = plt.subplots(2, 2, figsize=(12, 10))

        # Prepare data
        names = [m.name for m in metrics_list]
        colors = plt.cm.viridis(np.linspace(0, 1, len(metrics_list)))

        # 1. Mean trajectory error
        ax = axes[0, 0]
        mean_errors = [np.mean(m.trajectory_error) for m in metrics_list]
        ax.bar(names, mean_errors, color=colors)
        ax.set_ylabel('Mean Error (rad)')
        ax.set_title('Trajectory Tracking Error')
        ax.tick_params(axis='x', rotation=45)
        ax.grid(True, alpha=0.3)

        # 2. Smoothness
        ax = axes[0, 1]
        smoothness = [m.smoothness for m in metrics_list]
        ax.bar(names, smoothness, color=colors)
        ax.set_ylabel('Smoothness Score')
        ax.set_title('Movement Smoothness (higher is better)')
        ax.tick_params(axis='x', rotation=45)
        ax.grid(True, alpha=0.3)

        # 3. Energy expenditure
        ax = axes[1, 0]
        energy = [m.energy for m in metrics_list]
        ax.bar(names, energy, color=colors)
        ax.set_ylabel('Energy')
        ax.set_title('Energy Expenditure (lower is better)')
        ax.tick_params(axis='x', rotation=45)
        ax.grid(True, alpha=0.3)

        # 4. Success rate
        ax = axes[1, 1]
        success_rates = [m.success_rate * 100 for m in metrics_list]
        ax.bar(names, success_rates, color=colors)
        ax.set_ylabel('Success Rate (%)')
        ax.set_title('Target Reaching Success')
        ax.set_ylim([0, 105])
        ax.tick_params(axis='x', rotation=45)
        ax.grid(True, alpha=0.3)

        plt.tight_layout()
        return fig

    @staticmethod
    def plot_dimensionality_analysis(metrics_list):
        """Analyze performance vs dimensionality"""
        fig, axes = plt.subplots(1, 3, figsize=(15, 4))

        dims = [m.dimensionality for m in metrics_list]

        # Error vs dimensionality
        ax = axes[0]
        errors = [np.mean(m.trajectory_error) for m in metrics_list]
        ax.scatter(dims, errors, s=100, alpha=0.7)
        for i, m in enumerate(metrics_list):
            ax.annotate(m.name, (dims[i], errors[i]),
                       fontsize=8, alpha=0.7)
        ax.set_xlabel('Controller Dimensionality')
        ax.set_ylabel('Mean Error')
        ax.set_title('Accuracy vs Dimensionality')
        ax.grid(True, alpha=0.3)

        # Smoothness vs dimensionality
        ax = axes[1]
        smoothness = [m.smoothness for m in metrics_list]
        ax.scatter(dims, smoothness, s=100, alpha=0.7, color='green')
        for i, m in enumerate(metrics_list):
            ax.annotate(m.name, (dims[i], smoothness[i]),
                       fontsize=8, alpha=0.7)
        ax.set_xlabel('Controller Dimensionality')
        ax.set_ylabel('Smoothness')
        ax.set_title('Smoothness vs Dimensionality')
        ax.grid(True, alpha=0.3)

        # Energy vs dimensionality
        ax = axes[2]
        energy = [m.energy for m in metrics_list]
        ax.scatter(dims, energy, s=100, alpha=0.7, color='red')
        for i, m in enumerate(metrics_list):
            ax.annotate(m.name, (dims[i], energy[i]),
                       fontsize=8, alpha=0.7)
        ax.set_xlabel('Controller Dimensionality')
        ax.set_ylabel('Energy')
        ax.set_title('Energy vs Dimensionality')
        ax.grid(True, alpha=0.3)

        plt.tight_layout()
        return fig

## 7. Main Experiment Pipeline

In [ ]:
def run_experiment(gui=True, n_targets=5):
    """
    Run complete experimental pipeline

    Args:
        gui: Show PyBullet GUI (slower but visual)
        n_targets: Number of reaching targets
    """
    print("="*60)
    print("NEURAL MANIFOLD ROBOT CONTROL EXPERIMENT")
    print("="*60)

    # 1. Initialize robot
    print("\n[1/5] Initializing robot...")
    robot = Robot6DOF(gui=gui)
    robot.reset()

    # 2. Generate tasks
    print("\n[2/5] Generating tasks...")
    task_gen = TaskGenerator(robot)
    targets = task_gen.generate_reaching_sequence(n_targets)
    print(f"  Generated {len(targets)} reaching targets")

    # 3. Create controllers
    print("\n[3/5] Creating controllers...")
    controllers = [
        FullDimensionalController(robot),
        ManifoldController(robot, latent_dim=1, manifold_type='linear'),
        ManifoldController(robot, latent_dim=2, manifold_type='linear'),
        ManifoldController(robot, latent_dim=3, manifold_type='linear'),
        ManifoldController(robot, latent_dim=2, manifold_type='synergy'),
        ManifoldController(robot, latent_dim=3, manifold_type='pca'),
    ]
    print(f"  Created {len(controllers)} controllers")

    # 4. Run experiments
    print("\n[4/5] Running experiments...")
    metrics_list = []
    trajectories = {}

    for i, controller in enumerate(controllers):
        print(f"  Testing {controller.name}... ({i+1}/{len(controllers)})")
        robot.reset()

        # Evaluate
        metrics = PerformanceEvaluator.evaluate_controller(controller, targets)
        metrics_list.append(metrics)
        trajectories[controller.name] = controller.trajectory_history

        print(f"    ✓ Error: {np.mean(metrics.trajectory_error):.4f}, "
              f"Smoothness: {metrics.smoothness:.2f}, "
              f"Success: {metrics.success_rate*100:.1f}%")

    # 5. Visualize results
    print("\n[5/5] Generating visualizations...")

    fig1 = Visualizer.plot_trajectories(trajectories, targets)
    plt.savefig('simulations/trajectories.png', dpi=150, bbox_inches='tight')

    fig2 = Visualizer.plot_performance_comparison(metrics_list)
    plt.savefig('simulations/performance.png', dpi=150, bbox_inches='tight')

    fig3 = Visualizer.plot_dimensionality_analysis(metrics_list)
    plt.savefig('simulations/dimensionality.png', dpi=150, bbox_inches='tight')

    # Save metrics to JSON
    metrics_dict = {
        m.name: {
            'dimensionality': m.dimensionality,
            'mean_error': float(np.mean(m.trajectory_error)),
            'smoothness': float(m.smoothness),
            'energy': float(m.energy),
            'success_rate': float(m.success_rate),
            'completion_time': float(m.completion_time)
        }
        for m in metrics_list
    }

    with open('simulations/metrics.json', 'w') as f:
        json.dump(metrics_dict, f, indent=2)

    # Print summary table
    print("\n" + "="*60)
    print("RESULTS SUMMARY")
    print("="*60)

    df = pd.DataFrame([
        {
            'Controller': m.name,
            'Dim': m.dimensionality,
            'Error': f"{np.mean(m.trajectory_error):.4f}",
            'Smoothness': f"{m.smoothness:.2f}",
            'Energy': f"{m.energy:.1f}",
            'Success%': f"{m.success_rate*100:.1f}"
        }
        for m in metrics_list
    ])

    print(df.to_string(index=False))

    # Cleanup
    robot.disconnect()

    return metrics_list, trajectories

## 8. Run the Experiment

In [ ]:
# Run experiment without GUI for faster execution
# Set gui=True to visualize in PyBullet (slower)
metrics, trajectories = run_experiment(gui=True, n_targets=5)

## 9. Display Results

In [ ]:
# Display trajectory plots
from IPython.display import Image
display(Image('simulations/trajectories.png'))

In [ ]:
# Display performance comparison
display(Image('simulations/performance.png'))

In [ ]:
# Display dimensionality analysis
display(Image('simulations/dimensionality.png'))

## 10. Analysis and Discussion

In [ ]:
# Load and display detailed metrics
with open('simulations/metrics.json', 'r') as f:
    detailed_metrics = json.load(f)

print("DETAILED METRICS")
print("="*60)
for name, metrics in detailed_metrics.items():
    print(f"\n{name}:")
    for key, value in metrics.items():
        print(f"  {key}: {value}")

## Key Findings

This experiment tests whether low-dimensional manifold constraints can effectively control embodied robotic systems:

1. **Dimensional Compression**: Manifold controllers (1D-3D) operate in compressed latent spaces compared to full 6D control

2. **Performance Trade-offs**:
   - Higher dimensional controllers may achieve better accuracy
   - Lower dimensional controllers may show smoother, more coordinated movements
   - Energy efficiency varies with manifold structure

3. **Manifold Structure Matters**:
   - Linear manifolds: Simple random projections
   - Synergy manifolds: Coordinated joint patterns (bio-inspired)
   - PCA manifolds: Data-driven compression

4. **Biological Relevance**: This suggests that neural manifolds in motor cortex may indeed contribute to effective motor control, not just emerge as a byproduct of connectivity

## 11. Additional Experiments

In [ ]:
# Run with different number of targets
print("Testing with more targets...")
metrics_10, _ = run_experiment(gui=False, n_targets=10)

In [ ]:
# Test specific manifold types
def compare_manifold_types(latent_dim=2):
    """Compare different manifold structures at same dimensionality"""
    robot = Robot6DOF(gui=False)
    task_gen = TaskGenerator(robot)
    targets = task_gen.generate_reaching_sequence(5)

    manifold_types = ['linear', 'synergy', 'pca']
    results = {}

    for mtype in manifold_types:
        robot.reset()
        controller = ManifoldController(robot, latent_dim=latent_dim,
                                       manifold_type=mtype)
        metrics = PerformanceEvaluator.evaluate_controller(controller, targets)
        results[mtype] = metrics
        print(f"{mtype}: Error={np.mean(metrics.trajectory_error):.4f}, "
              f"Smoothness={metrics.smoothness:.2f}")

    robot.disconnect()
    return results

print(f"\nComparing manifold types at 2D:")
manifold_comparison = compare_manifold_types(latent_dim=2)